# Danh Sách Cán Bộ - Giáo Viên - Nhân Viên

Notebook này đọc dữ liệu từ file Excel, chuyển đổi sang JSON trong bộ nhớ và hiển thị HTML danh sách cán bộ giáo viên.

## 1. Import thư viện cần thiết

In [120]:
import pandas as pd
import datetime
from IPython.display import HTML, display
import json

## 2. Đọc dữ liệu từ file Excel

Chỉ định đường dẫn file Excel và đọc dữ liệu vào DataFrame

In [121]:
# Đường dẫn file Excel - thay đổi nếu cần
excel_file_path = 'DS GV.xlsx'

# Đọc file Excel
df = pd.read_excel(excel_file_path, engine='openpyxl')

print(f"Đã đọc {len(df)} dòng dữ liệu")
print(f"Các cột: {list(df.columns)}")

Đã đọc 101 dòng dữ liệu
Các cột: ['STT', 'Tổ', 'Họ tên', 'Giới tính', 'Ngày sinh', 'Email', 'Điện thoại', 'Là Đảng viên', 'Vị trí việc làm', 'Nhóm chức vụ', 'Môn dạy']


## 3. Chuyển đổi sang JSON và xem trước

Chuyển DataFrame sang định dạng JSON trong bộ nhớ

In [122]:
# Chuyển đổi sang JSON (trong bộ nhớ, không ghi file)
data = df.to_dict('records')

# Xem trước 3 dòng đầu
print("Xem trước 3 dòng đầu:\n")
print(json.dumps(data[:3], indent=2, ensure_ascii=False))

Xem trước 3 dòng đầu:

[
  {
    "STT": 1,
    "Tổ": "Ban giám hiệu",
    "Họ tên": "Nguyễn Văn Định",
    "Giới tính": "Nam",
    "Ngày sinh": "16/03/1975",
    "Email": "nvdinh.c3tcvan@khanhhoa.edu.vn",
    "Điện thoại": 868663998,
    "Là Đảng viên": "x",
    "Vị trí việc làm": "Cán bộ quản lý",
    "Nhóm chức vụ": "Phó hiệu trưởng phụ trách",
    "Môn dạy": "Tiếng Anh"
  },
  {
    "STT": 2,
    "Tổ": "Ban giám hiệu",
    "Họ tên": "Lê Thanh Tuấn",
    "Giới tính": "Nam",
    "Ngày sinh": "12/07/1982",
    "Email": "lttuan.c3tcvan@khanhhoa.edu.vn",
    "Điện thoại": 932139777,
    "Là Đảng viên": "x",
    "Vị trí việc làm": "Cán bộ quản lý",
    "Nhóm chức vụ": "Phó hiệu trưởng",
    "Môn dạy": "Toán"
  },
  {
    "STT": 1,
    "Tổ": "Tổ GDTC - QPAN",
    "Họ tên": "Lê Văn Tiên",
    "Giới tính": "Nam",
    "Ngày sinh": "06/05/1987",
    "Email": "lvtien.c3tcvan@khanhhoa.edu.vn",
    "Điện thoại": 367208422,
    "Là Đảng viên": NaN,
    "Vị trí việc làm": "Giáo viên",
    "Nhóm chứ

## 4. Nhóm dữ liệu theo Tổ

Phân nhóm cán bộ giáo viên theo bộ môn

In [123]:
# Nhóm theo Tổ
groups = {}
for person in data:
    group_name = person.get('Tổ', 'Khác')
    if group_name not in groups:
        groups[group_name] = []
    groups[group_name].append(person)

sorted_group_names = sorted(groups.keys())

print(f"Tổng số tổ: {len(groups)}")
for group_name in sorted_group_names:
    print(f"  - {group_name}: {len(groups[group_name])} người")

Tổng số tổ: 11
  - Ban giám hiệu: 2 người
  - Giáo viên thỉnh giảng: 1 người
  - Tổ GDTC - QPAN: 7 người
  - Tổ Hoá: 11 người
  - Tổ Lý - Công nghệ: 13 người
  - Tổ Sinh - Tin: 12 người
  - Tổ Tiếng Anh: 11 người
  - Tổ Toán: 13 người
  - Tổ Văn: 13 người
  - Tổ Văn phòng: 6 người
  - Tổ Xã hội: 12 người


## 5. Tạo HTML hiển thị danh sách

Tạo HTML với Bootstrap styling (trong bộ nhớ)

In [124]:
def convert_excel_date(dob):
    """Chuyển đổi ngày tháng từ Excel serial date sang dd/mm/yyyy"""
    if isinstance(dob, int):
        try:
            d = datetime.datetime(1899, 12, 30) + datetime.timedelta(days=dob)
            return d.strftime('%d/%m/%Y')
        except:
            return str(dob)
    elif pd.notna(dob):
        return str(dob)
    return ''

def get_vietnamese_name(full_name):
    """Lấy tên (từ cuối cùng) từ họ tên đầy đủ"""
    if pd.notna(full_name) and str(full_name).strip():
        parts = str(full_name).strip().split()
        if parts:
            return parts[-1]  # Tên là từ cuối cùng
    return ""

def vietnamese_sort_key(text):
    """Tạo key để sắp xếp tiếng Việt theo thứ tự alphabet"""
    # Bảng chuyển đổi các ký tự tiếng Việt sang thứ tự sắp xếp
    vietnamese_order = {
        'a': 'a', 'á': 'a', 'à': 'a', 'ả': 'a', 'ã': 'a', 'ạ': 'a',
        'ă': 'a1', 'ắ': 'a1', 'ằ': 'a1', 'ẳ': 'a1', 'ẵ': 'a1', 'ặ': 'a1',
        'â': 'a2', 'ấ': 'a2', 'ầ': 'a2', 'ẩ': 'a2', 'ẫ': 'a2', 'ậ': 'a2',
        'b': 'b', 'c': 'c', 'd': 'd', 'đ': 'd1',
        'e': 'e', 'é': 'e', 'è': 'e', 'ẻ': 'e', 'ẽ': 'e', 'ẹ': 'e',
        'ê': 'e1', 'ế': 'e1', 'ề': 'e1', 'ể': 'e1', 'ễ': 'e1', 'ệ': 'e1',
        'f': 'f', 'g': 'g', 'h': 'h',
        'i': 'i', 'í': 'i', 'ì': 'i', 'ỉ': 'i', 'ĩ': 'i', 'ị': 'i',
        'j': 'j', 'k': 'k', 'l': 'l', 'm': 'm', 'n': 'n',
        'o': 'o', 'ó': 'o', 'ò': 'o', 'ỏ': 'o', 'õ': 'o', 'ọ': 'o',
        'ô': 'o1', 'ố': 'o1', 'ồ': 'o1', 'ổ': 'o1', 'ỗ': 'o1', 'ộ': 'o1',
        'ơ': 'o2', 'ớ': 'o2', 'ờ': 'o2', 'ở': 'o2', 'ỡ': 'o2', 'ợ': 'o2',
        'p': 'p', 'q': 'q', 'r': 'r', 's': 's', 't': 't',
        'u': 'u', 'ú': 'u', 'ù': 'u', 'ủ': 'u', 'ũ': 'u', 'ụ': 'u',
        'ư': 'u1', 'ứ': 'u1', 'ừ': 'u1', 'ử': 'u1', 'ữ': 'u1', 'ự': 'u1',
        'v': 'v', 'w': 'w', 'x': 'x',
        'y': 'y', 'ý': 'y', 'ỳ': 'y', 'ỷ': 'y', 'ỹ': 'y', 'ỵ': 'y',
        'z': 'z'
    }
    
    result = ""
    for char in text.lower():
        result += vietnamese_order.get(char, char)
    return result

def sort_members(members):
    """Sắp xếp thành viên: Tổ trưởng -> Tổ phó -> Các thành viên khác (theo tên)"""
    to_truong = []
    to_pho = []
    others = []
    
    for person in members:
        role_group = person.get('Nhóm chức vụ', '')
        role_str = str(role_group).lower() if pd.notna(role_group) else ''
        
        if 'tổ trưởng' in role_str or 'trưởng' in role_str:
            to_truong.append(person)
        elif 'tổ phó' in role_str or 'phó' in role_str:
            to_pho.append(person)
        else:
            others.append(person)
    
    # Sắp xếp các thành viên khác theo tên (từ cuối cùng) với thứ tự tiếng Việt
    others.sort(key=lambda x: vietnamese_sort_key(get_vietnamese_name(x.get('Họ tên', ''))))
    
    return to_truong + to_pho + others

def sort_group_names(group_names):
    """Sắp xếp tên tổ: BGH -> các tổ khác (A-Z) -> GV thỉnh giảng"""
    bgh = []
    gv_thinh_giang = []
    others = []
    
    for name in group_names:
        name_lower = name.lower()
        if 'ban giám hiệu' in name_lower or 'bgh' in name_lower:
            bgh.append(name)
        elif 'thỉnh giảng' in name_lower or 'thinh giang' in name_lower:
            gv_thinh_giang.append(name)
        else:
            others.append(name)
    
    # Sắp xếp các tổ khác theo tên
    others.sort()
    
    return bgh + others + gv_thinh_giang

# Tạo HTML với bảng đơn giản
html_content = """<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Danh sách Cán bộ - Giáo viên - Nhân viên</title>
    <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.4.1/css/bootstrap.min.css">
</head>
<body style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 15px; margin: 0; min-height: 100vh;">
    <div class="container-fluid" style="max-width: 1400px; margin: 0 auto; background-color: white; padding: 25px; box-shadow: 0 10px 40px rgba(0,0,0,0.2); border-radius: 10px;">
        <h1 style="text-align: center; color: #2c3e50; margin-bottom: 25px; font-size: 32px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px;">Danh Sách Cán Bộ - Giáo Viên - Nhân Viên</h1>
"""

# Sắp xếp tên các tổ: BGH -> các tổ khác -> GV thỉnh giảng
sorted_group_names_custom = sort_group_names(sorted_group_names)

for group_name in sorted_group_names_custom:
    members = groups[group_name]
    sorted_members = sort_members(members)
    
    html_content += f"""
        <div style="margin-top: 30px; margin-bottom: 20px;">
            <h2 style="background: linear-gradient(to right, #3357F5FF, #5F229DFF); -webkit-background-clip: text; -webkit-text-fill-color: transparent; background-clip: text; font-size: 24px; display: inline-block; margin: 0 0 15px 0; font-weight: bold;">
            {group_name} <small>({len(members)} người)</small>
            </h2>
            
        </div>
        <table class="table table-bordered table-hover" style="margin-bottom: 30px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
            <thead>
                <tr style="background: linear-gradient(to right, #B5C1F8FF, #CFA7F7FF) !important; color: black;">
                    <th style="width: 5%; padding: 12px; text-align: center;">STT</th>
                    <th style="width: 18%; padding: 12px; text-align: center;">Họ và tên</th>
                    <th style="width: 11%; text-align: center; padding: 12px;">Ngày sinh</th>
                    <th style="width: 18%; padding: 12px; text-align: center;">Chức vụ</th>
                    <th style="width: 23%; padding: 12px; text-align: center;">Môn dạy</th>
                    <th style="width: 13%; text-align: center; padding: 12px;">Liên hệ</th>
                    <th style="width: 8%; text-align: center; padding: 12px;">Đảng viên</th>
                </tr>
            </thead>
            <tbody>
    """
    
    for idx, person in enumerate(sorted_members, 1):
        gender = person.get('Giới tính', '')
        
        # Set colors based on gender
        if gender == 'Nam':
            row_bg = '#e8f4f8'
            name_color = '#2c5aa0'
        else:
            row_bg = '#ffe8f0'
            name_color = '#c41e3a'
            
        name = person.get('Họ tên', '')
        dob = convert_excel_date(person.get('Ngày sinh', ''))
        
        # Xử lý chức vụ
        job_title = person.get('Vị trí việc làm', '')
        role_group = person.get('Nhóm chức vụ', '')
        
        if pd.notna(role_group) and str(role_group).strip() and str(role_group).lower() != 'nan':
            chu_vu = str(role_group)
        elif pd.notna(job_title) and str(job_title).strip() and str(job_title).lower() != 'nan':
            chu_vu = str(job_title)
        else:
            chu_vu = ""
        
        # Xử lý môn dạy
        subject = person.get('Môn dạy', '')
        if pd.notna(subject) and str(subject).strip() and str(subject).lower() != 'nan':
            mon_day = str(subject)
        else:
            mon_day = ""
            
        party_member = person.get('Là Đảng viên', '')
        party_star = '★' if party_member == 'x' else ''
        
        email = person.get('Email', '')
        phone_raw = person.get('Điện thoại', '')
        phone = ''
        if pd.notna(phone_raw):
            phone = str(int(phone_raw)) if isinstance(phone_raw, float) else str(phone_raw)
            # Thêm số 0 vào đầu nếu thiếu (số điện thoại VN có 10 chữ số)
            if phone and phone.isdigit() and len(phone) == 9:
                phone = '0' + phone
        
        # Contact icons
        contact_icons = []
        if email:
            contact_icons.append(f'<a href="mailto:{email}" title="{email}" style="text-decoration: none; font-size: 20px; color: #3498db; margin-right: 8px;">✉</a>')
        if phone:
            contact_icons.append(f'<a href="tel:{phone}" title="{phone}" style="text-decoration: none; font-size: 20px; color: #27ae60;">☎</a>')
            
        contact_html = "".join(contact_icons) if contact_icons else '<span style="color: #999; font-size: 12px;">—</span>'
        
        html_content += f"""
                <tr style="background-color: {row_bg};">
                    <td style="padding: 10px; text-align: center; font-weight: bold;">{idx}</td>
                    <td style="padding: 10px; font-weight: bold; color: {name_color}; text-align: center;">{name}</td>
                    <td style="text-align: center; padding: 10px;">{dob}</td>
                    <td style="padding: 10px; text-align: center;">{chu_vu}</td>
                    <td style="padding: 10px; text-align: center;">{mon_day}</td>
                    <td style="text-align: center; padding: 10px;">{contact_html}</td>
                    <td style="text-align: center; padding: 10px; font-size: 18px; color: #e74c3c;">{party_star}</td>
                </tr>
        """
        
    html_content += """
            </tbody>
        </table>
    """

html_content += """
    </div>
    <footer style="text-align: center; padding: 20px; color: white; margin-top: 25px; font-size: 13px;">
        <p style="margin: 0; text-shadow: 1px 1px 2px rgba(0,0,0,0.3);">© 2025 Trường THPT TCV</p>
    </footer>
</body>
</html>
"""

print("HTML đã được tạo với thứ tự: BGH → Các tổ → GV thỉnh giảng")

HTML đã được tạo với thứ tự: BGH → Các tổ → GV thỉnh giảng


## 6. Hiển thị HTML trong Notebook

Render HTML trực tiếp trong notebook (không ghi ra file)

In [125]:
# Hiển thị HTML trực tiếp trong notebook
display(HTML(html_content))

STT,Họ và tên,Ngày sinh,Chức vụ,Môn dạy,Liên hệ,Đảng viên
1,Nguyễn Văn Định,16/03/1975,Phó hiệu trưởng phụ trách,Tiếng Anh,✉☎,★
2,Lê Thanh Tuấn,12/07/1982,Phó hiệu trưởng,Toán,✉☎,★
STT,Họ và tên,Ngày sinh,Chức vụ,Môn dạy,Liên hệ,Đảng viên
1,Trịnh Quốc Đạt,30/10/1981,Tổ trưởng chuyên môn,Thể dục,✉☎,★
2,Nguyễn Văn Thành,30/06/1986,Tổ phó chuyên môn,Thể dục,✉☎,
3,Nguyễn Thái Cường,28/04/1989,Giáo viên,Thể dục,✉☎,
4,Nguyễn Văn Nhã,10/09/1979,Giáo viên,Thể dục,✉☎,
5,Lê Văn Tiên,06/05/1987,Giáo viên,Giáo dục thể chất,✉☎,
6,Tạ Thanh Toàn,28/02/1978,Giáo viên,Giáo dục thể chất,✉☎,★
7,Nguyễn Thanh Tuấn,26/06/1994,Giáo viên,Giáo dục thể chất,✉☎,


In [126]:
with open('GV.html', 'w', encoding='utf-8') as f:
    f.write(html_content)